# OTE PublicDataService → SQLite

This notebook downloads public OTE day-ahead market electricity data from the public SOAP service and stores the results into SQLite.

It is designed to work **without login** against the public namespace:

- `http://www.ote-cr.cz/schema/service/public`
- service endpoint typically documented as `http://www.ote-cr.cz/services/PublicDataService`

The notebook uses `GetDamPriceE`, and the parser is intentionally tolerant:
- it handles the historically documented **hourly** structure (`Date`, `Hour`, `Price`, `Volume`)
- it also tries to capture **15-minute** style fields if they appear in the current public response

> Update the `START_DATE`, `END_DATE`, and database path in the configuration cell before running.


## What gets stored

The SQLite table `ote_day_ahead_prices` stores:

- `delivery_date`
- `hour`
- `quarter`
- `period_start`
- `period_end`
- `resolution_min`
- `price`
- `volume`
- `currency`
- `source_method`
- `raw_item_xml`

This allows mixing hourly and 15-minute rows in one table.


In [1]:

import requests
import sqlite3
import pandas as pd
from datetime import date, datetime, timedelta
from xml.etree import ElementTree as ET
from pathlib import Path
from typing import Optional, List, Dict

# -----------------------------
# Configuration
# -----------------------------
SERVICE_URL = "http://www.ote-cr.cz/services/PublicDataService"
SOAP_NS = "http://schemas.xmlsoap.org/soap/envelope/"
PUB_NS = "http://www.ote-cr.cz/schema/service/public"

START_DATE = "2025-10-01"   # inclusive
END_DATE   = "2025-10-07"   # inclusive
IN_EUR = True

SQLITE_PATH = Path("FVEDB.db")
TABLE_NAME = "SpotData"

REQUEST_TIMEOUT = 60

print("Service URL:", SERVICE_URL)
print("Date range :", START_DATE, "->", END_DATE)
print("SQLite DB  :", SQLITE_PATH.resolve())


Service URL: http://www.ote-cr.cz/services/PublicDataService
Date range : 2025-10-01 -> 2025-10-07
SQLite DB  : C:\Users\hp\Documents\TUL\DP\Predikce výroby\FVEDB.db


In [2]:

# Optional: quick connectivity check
wsdl_url = SERVICE_URL.rstrip("/") + "/wsdl"
try:
    r = requests.get(wsdl_url, timeout=REQUEST_TIMEOUT)
    print("WSDL status:", r.status_code)
    print(r.text[:300])
except Exception as e:
    print("WSDL check failed:", e)


WSDL status: 200
<?xml version="1.0" encoding="utf-8" ?>
<wsdl:definitions xmlns:http="http://schemas.xmlsoap.org/wsdl/http/" xmlns:soap="http://schemas.xmlsoap.org/wsdl/soap/" xmlns:xs="http://www.w3.org/2001/XMLSchema" xmlns:soapenc="http://schemas.xmlsoap.org/soap/encoding/" xmlns:tns="http://www.ote-cr.cz/schema


In [3]:

def iso_dates(start_date: str, end_date: str):
    d0 = date.fromisoformat(start_date)
    d1 = date.fromisoformat(end_date)
    d = d0
    while d <= d1:
        yield d.isoformat()
        d += timedelta(days=1)

def build_get_dam_price_envelope(start_date: str, end_date: str, start_hour: int = 1, end_hour: int = 24, in_eur: bool = True) -> str:
    in_eur_str = "true" if in_eur else "false"
    return f'''<?xml version="1.0" encoding="utf-8"?>
<soapenv:Envelope xmlns:soapenv="{SOAP_NS}" xmlns:pub="{PUB_NS}">
  <soapenv:Header/>
  <soapenv:Body>
    <pub:GetDamPriceE>
      <pub:StartDate>{start_date}</pub:StartDate>
      <pub:EndDate>{end_date}</pub:EndDate>
      <pub:StartHour>{start_hour}</pub:StartHour>
      <pub:EndHour>{end_hour}</pub:EndHour>
      <pub:InEur>{in_eur_str}</pub:InEur>
    </pub:GetDamPriceE>
  </soapenv:Body>
</soapenv:Envelope>'''

def call_get_dam_price(start_date: str, end_date: str, start_hour: int = 1, end_hour: int = 24, in_eur: bool = True) -> str:
    xml_body = build_get_dam_price_envelope(start_date, end_date, start_hour, end_hour, in_eur)
    headers = {
        "Content-Type": "text/xml; charset=utf-8",
        # SOAPAction is usually not required here, but empty value is often accepted.
        "SOAPAction": '""',
    }
    resp = requests.post(SERVICE_URL, data=xml_body.encode("utf-8"), headers=headers, timeout=REQUEST_TIMEOUT)
    resp.raise_for_status()
    return resp.text


In [4]:

def strip_ns(tag: str) -> str:
    return tag.split("}", 1)[1] if "}" in tag else tag

def text_or_none(elem: Optional[ET.Element]) -> Optional[str]:
    if elem is None or elem.text is None:
        return None
    txt = elem.text.strip()
    return txt if txt != "" else None

def parse_float(value: Optional[str]) -> Optional[float]:
    if value is None:
        return None
    value = value.replace(",", ".")
    try:
        return float(value)
    except Exception:
        return None

def infer_resolution_and_bounds(delivery_date: Optional[str], hour: Optional[int], quarter: Optional[int]):
    # Hour-only historical public output -> 60 min period
    if delivery_date and hour is not None and quarter is None:
        start_dt = datetime.fromisoformat(f"{delivery_date}T00:00:00") + timedelta(hours=hour - 1)
        end_dt = start_dt + timedelta(hours=1)
        return start_dt.isoformat(sep=" "), end_dt.isoformat(sep=" "), 60

    # Hour + quarter -> 15 min
    if delivery_date and hour is not None and quarter is not None:
        start_dt = datetime.fromisoformat(f"{delivery_date}T00:00:00") + timedelta(hours=hour - 1, minutes=(quarter - 1) * 15)
        end_dt = start_dt + timedelta(minutes=15)
        return start_dt.isoformat(sep=" "), end_dt.isoformat(sep=" "), 15

    return None, None, None

def parse_result_items(xml_text: str, source_method: str = "GetDamPriceE") -> pd.DataFrame:
    root = ET.fromstring(xml_text.encode("utf-8"))
    rows: List[Dict] = []

    for elem in root.iter():
        if strip_ns(elem.tag) != "Item":
            continue

        children = {strip_ns(ch.tag): (ch.text.strip() if ch.text else None) for ch in list(elem)}

        delivery_date = children.get("Date")
        hour_raw = children.get("Hour")
        quarter_raw = children.get("Quarter") or children.get("QH") or children.get("QuarterHour")

        hour = int(hour_raw) if hour_raw and hour_raw.isdigit() else None
        quarter = int(quarter_raw) if quarter_raw and quarter_raw.isdigit() else None

        # Flexible capture of period boundaries, if the newer service returns them explicitly
        period_start = (
            children.get("PeriodStart")
            or children.get("Start")
            or children.get("IntervalStart")
            or children.get("From")
        )
        period_end = (
            children.get("PeriodEnd")
            or children.get("End")
            or children.get("IntervalEnd")
            or children.get("To")
        )

        resolution_min = None
        if period_start and period_end:
            try:
                dt0 = datetime.fromisoformat(period_start.replace("Z", "+00:00"))
                dt1 = datetime.fromisoformat(period_end.replace("Z", "+00:00"))
                resolution_min = int((dt1 - dt0).total_seconds() // 60)
                period_start = dt0.isoformat(sep=" ")
                period_end = dt1.isoformat(sep=" ")
            except Exception:
                pass

        if resolution_min is None:
            auto_start, auto_end, auto_res = infer_resolution_and_bounds(delivery_date, hour, quarter)
            period_start = period_start or auto_start
            period_end = period_end or auto_end
            resolution_min = auto_res

        price = parse_float(children.get("Price") or children.get("PriceCZ") or children.get("PriceEur"))
        volume = parse_float(children.get("Volume") or children.get("VolumeCZ") or children.get("VolumeEur"))

        rows.append({
            "delivery_date": delivery_date,
            "hour": hour,
            "quarter": quarter,
            "period_start": period_start,
            "period_end": period_end,
            "resolution_min": resolution_min,
            "price": price,
            "volume": volume,
            "currency": "EUR" if IN_EUR else "CZK",
            "source_method": source_method,
            "raw_item_xml": ET.tostring(elem, encoding="unicode"),
        })

    return pd.DataFrame(rows)


In [ ]:
def insert_dataframe(df: pd.DataFrame, db_path: Path, table_name: str = TABLE_NAME):
    if df.empty:
        return 0
    with sqlite3.connect(db_path) as conn:
        df.to_sql(table_name, conn, if_exists="append", index=False)
    return len(df)

print("Database is ready.")


In [ ]:

# Download the whole requested interval in one call
raw_xml = call_get_dam_price(
    start_date=START_DATE,
    end_date=END_DATE,
    start_hour=1,
    end_hour=24,
    in_eur=IN_EUR,
)

print(raw_xml[:1500])


In [ ]:

df = parse_result_items(raw_xml, source_method="GetDamPriceE")
print("Rows parsed:", len(df))
df.head(20)


In [ ]:

inserted = insert_dataframe(df, SQLITE_PATH, TABLE_NAME)
print(f"Inserted rows: {inserted}")


In [ ]:

# Read back from SQLite
with sqlite3.connect(SQLITE_PATH) as conn:
    preview = pd.read_sql_query(
        f"SELECT DateTime, Value FROM {TABLE_NAME} ORDER BY dt LIMIT 50",
        conn
    )

preview


In [ ]:

# Optional helper: replace existing rows for the selected period before reloading
def delete_range(db_path: Path, start_date: str, end_date: str, table_name: str = TABLE_NAME):
    with sqlite3.connect(db_path) as conn:
        cur = conn.execute(
            f"DELETE FROM {table_name} WHERE delivery_date >= ? AND delivery_date <= ?",
            (start_date, end_date),
        )
        conn.commit()
        return cur.rowcount

# Example:
# deleted = delete_range(SQLITE_PATH, START_DATE, END_DATE)
# print("Deleted:", deleted)


## Notes

- This notebook intentionally uses plain `requests` and raw SOAP XML, so it works in a minimal Jupyter environment.
- The historical public OTE manual documents `GetDamPriceE` with hourly `Item` elements (`Date`, `Hour`, `Price`, `Volume`).
- Because OTE later announced public web services would expose both **15-minute** and **60-minute** day-ahead prices, the parser also looks for quarter-hour or explicit interval fields and stores the inferred `resolution_min`.
- If your live response contains additional fields, you can extend `parse_result_items()` easily.
